# 🌾 End-to-End Soil Image Classification & Precision Crop Recommendation Pipeline
### Machine Learning & Computer Vision System for Precision Agriculture
**Author:** Rashi Koiri  
**Dataset 1:** Soil Image Dataset (4 Classes: Alluvial, Black, Clay, Red Soil)  
**Dataset 2:** Soil Chemistry & Environmental Telemetry (2,200 records, 22 crop classes)  

---
### 📌 Workflow Overview
1. **Computer Vision (CNN / Transfer Learning - MobileNetV2):** Takes a photograph of soil as input and predicts the **Soil Type** with confidence.
2. **Agronomic Mapping Bridge:** Automatically derives estimated **Soil Macronutrients ($N, P, K$) and Reaction ($pH$)** based on the predicted soil type.
3. **Machine Learning Classifier (Random Forest):** Takes the derived $(N, P, K, pH)$ and user-provided environmental telemetry ($Temperature, Humidity, Rainfall$) to recommend the **Optimal Crop** to cultivate.


## 1. Setup Environment & Dataset Extraction
Upload your `archive (1).zip` (the soil images dataset) and `Crop_recommendation.csv` directly into Google Colab's left file pane, or mount Google Drive.


In [ ]:
# Step 1: Verify GPU & Install Dependencies
import os
import zipfile
import shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout, BatchNormalization
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import joblib

print("TensorFlow Version:", tf.__version__)
print("GPU Available:", tf.config.list_physical_devices('GPU'))


In [ ]:
# Step 2: Unzip Soil Image Dataset
zip_path = "archive (1).zip" # Or path to your zip file
extract_path = "soil_dataset"

if os.path.exists(zip_path):
    print("Extracting soil image dataset...")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_path)
    print("Extraction completed!")
else:
    print(f"Note: Please upload '{zip_path}' to the Colab files pane on the left!")

# Locate the dataset folder
if os.path.exists("soil_dataset/Dataset"):
    BASE_DIR = "soil_dataset/Dataset"
elif os.path.exists("Dataset"):
    BASE_DIR = "Dataset"
else:
    BASE_DIR = extract_path

TRAIN_DIR = os.path.join(BASE_DIR, "Train")
TEST_DIR = os.path.join(BASE_DIR, "test")

print("Train Directory:", TRAIN_DIR, "| Exists:", os.path.exists(TRAIN_DIR))
print("Test Directory:", TEST_DIR, "| Exists:", os.path.exists(TEST_DIR))


## 2. Exploratory Data Analysis (EDA) on Soil Images
Let's inspect the distribution of soil classes and display representative visual samples from each category.


In [ ]:
# Inspect Class Distribution
classes = sorted(os.listdir(TRAIN_DIR)) if os.path.exists(TRAIN_DIR) else ['Alluvial soil', 'Black Soil', 'Clay soil', 'Red soil']
print("Detected Soil Classes (4):", classes)

train_counts = {}
for cls in classes:
    cls_path = os.path.join(TRAIN_DIR, cls)
    if os.path.exists(cls_path):
        train_counts[cls] = len([f for f in os.listdir(cls_path) if f.lower().endswith(('.jpg', '.jpeg', '.png'))])

plt.figure(figsize=(9, 4))
sns.barplot(x=list(train_counts.keys()), y=list(train_counts.values()), palette='viridis')
plt.title("Soil Image Training Dataset Class Distribution", fontsize=14, fontweight='bold')
plt.ylabel("Number of Images")
plt.xlabel("Soil Type")
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()


In [ ]:
# Display Sample Soil Images
plt.figure(figsize=(14, 4))
for idx, cls in enumerate(classes):
    cls_path = os.path.join(TRAIN_DIR, cls)
    if os.path.exists(cls_path):
        sample_file = os.listdir(cls_path)[0]
        img_path = os.path.join(cls_path, sample_file)
        img = Image.open(img_path)
        plt.subplot(1, 4, idx + 1)
        plt.imshow(img)
        plt.title(f"{cls}\n({img.size[0]}x{img.size[1]})", fontweight='bold')
        plt.axis('off')
plt.tight_layout()
plt.show()


## 3. Image Data Preprocessing & Augmentation
We apply real-time data augmentation (horizontal flip, slight rotation, shear, zoom) to prevent overfitting on the soil texture patterns.


In [ ]:
# Image Hyperparameters
IMG_SIZE = (224, 224)
BATCH_SIZE = 32

train_datagen = ImageDataGenerator(
    rescale=1.0 / 255.0,
    rotation_range=20,
    width_shift_range=0.1,
    height_shift_range=0.1,
    shear_range=0.1,
    zoom_range=0.15,
    horizontal_flip=True,
    fill_mode='nearest',
    validation_split=0.2 # 20% validation split from training set
)

test_datagen = ImageDataGenerator(rescale=1.0 / 255.0)

train_generator = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='training',
    shuffle=True
)

val_generator = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='validation',
    shuffle=False
)

test_generator = test_datagen.flow_from_directory(
    TEST_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

class_indices = train_generator.class_indices
class_names = {v: k for k, v in class_indices.items()}
print("Class Indices Mapping:", class_indices)


## 4. CNN Model Architecture (Transfer Learning with MobileNetV2)
We employ **MobileNetV2** pre-trained on ImageNet. Its lightweight depthwise separable convolutions make it ultra-fast and exceptionally accurate for texture and grain recognition.


In [ ]:
# Build Transfer Learning Model
def build_soil_classifier(num_classes=4):
    base_model = MobileNetV2(
        weights='imagenet',
        include_top=False,
        input_shape=(224, 224, 3)
    )
    # Freeze base model features initially
    base_model.trainable = False

    x = base_model.output
    x = GlobalAveragePooling2D()(x)
    x = BatchNormalization()(x)
    x = Dense(256, activation='relu')(x)
    x = Dropout(0.4)(x)
    predictions = Dense(num_classes, activation='softmax')(x)

    model = Model(inputs=base_model.input, outputs=predictions)
    return model

soil_model = build_soil_classifier(num_classes=len(class_indices))
soil_model.compile(
    optimizer=Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

soil_model.summary()


## 5. Model Training & Evaluation
We train the model with `EarlyStopping` and `ReduceLROnPlateau` callbacks to achieve maximum validation accuracy without overfitting.


In [ ]:
# Define Callbacks
callbacks = [
    EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=3, min_lr=1e-6, verbose=1),
    ModelCheckpoint('best_soil_model.keras', monitor='val_accuracy', save_best_only=True, verbose=1)
]

# Train
EPOCHS = 15
print("Starting Soil CNN Training...")
history = soil_model.fit(
    train_generator,
    epochs=EPOCHS,
    validation_data=val_generator,
    callbacks=callbacks
)
print("Training completed!")


In [ ]:
# Plot Training & Validation Curves
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Train Accuracy', color='#16a34a', linewidth=2)
plt.plot(history.history['val_accuracy'], label='Val Accuracy', color='#0284c7', linewidth=2)
plt.title('Soil Classification Accuracy Curve', fontweight='bold')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.6)

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Train Loss', color='#dc2626', linewidth=2)
plt.plot(history.history['val_loss'], label='Val Loss', color='#ea580c', linewidth=2)
plt.title('Soil Classification Loss Curve', fontweight='bold')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.6)

plt.tight_layout()
plt.show()


In [ ]:
# Evaluate on Test Set
test_loss, test_acc = soil_model.evaluate(test_generator)
print(f"\n🎯 Final Test Accuracy on Unseen Soil Images: {test_acc * 100:.2f}%")

# Confusion Matrix
Y_pred = soil_model.predict(test_generator)
y_pred = np.argmax(Y_pred, axis=1)
y_true = test_generator.classes

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens', xticklabels=class_names.values(), yticklabels=class_names.values())
plt.title('Soil Classification Confusion Matrix', fontweight='bold')
plt.ylabel('Actual Soil Class')
plt.xlabel('Predicted Soil Class')
plt.show()

print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=class_names.values()))


## 6. Crop Recommendation Model (Random Forest on Tabular Data)
We train the Random Forest Classifier on `Crop_recommendation.csv` (2,200 records, 7 input features, 22 crops).


In [ ]:
# Load Crop Recommendation Dataset
csv_file = "Crop_recommendation.csv"
if not os.path.exists(csv_file):
    print("Warning: Crop_recommendation.csv not found in current directory. Please upload it!")

df_crop = pd.read_csv(csv_file)
print("Crop Dataset Shape:", df_crop.shape)

X = df_crop[['N', 'P', 'K', 'temperature', 'humidity', 'ph', 'rainfall']]
y = df_crop['label']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f"Crop Training samples: {X_train.shape[0]}, Testing samples: {X_test.shape[0]}")

rf_crop_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_crop_model.fit(X_train, y_train)

y_pred_crop = rf_crop_model.predict(X_test)
crop_acc = accuracy_score(y_test, y_pred_crop)
print(f"🎯 Crop Recommendation Test Accuracy: {crop_acc * 100:.2f}%")

# Save trained models
soil_model.save("soil_classifier_model.keras")
joblib.dump(rf_crop_model, "crop_recommendation_model.pkl")
print("Saved both models successfully!")


## 7. The Agronomic Bridge: Mapping Soil Type to Estimated N-P-K & pH
Based on agricultural soil chemistry benchmarks (Indian Council of Agricultural Research & Field Telemetry), each soil class exhibits a characteristic macronutrient profile and soil reaction.


In [ ]:
# Agronomic Benchmark Matrix for the 4 Soil Classes
SOIL_NUTRIENT_MAPPING = {
    "Alluvial soil": {
        "N": 90.0,    # Rich in organic nitrogen and potash
        "P": 42.0,    # Moderate phosphorus
        "K": 43.0,    # Adequate potassium
        "ph": 6.5,    # Slightly acidic to neutral
        "soil_desc": "Alluvial River Plain Loam (Highly fertile, optimal for paddy, jute, wheat)"
    },
    "Black Soil": {
        "N": 118.0,   # High nitrogen retention
        "P": 48.0,    # Moderate available phosphorus
        "K": 20.0,    # High montmorillonite clay content, calcium-rich
        "ph": 7.6,    # Moderately alkaline reaction
        "soil_desc": "Deccan Regur / Vertisol (Self-aerating clay, optimal for cotton, pigeon peas)"
    },
    "Clay soil": {
        "N": 78.0,    # Dense nutrient concentration
        "P": 58.0,    # High phosphorus fixation
        "K": 52.0,    # High potassium retention
        "ph": 6.3,    # Slightly acidic
        "soil_desc": "Heavy Clay Loam (High water table retention, ideal for rice, lentils)"
    },
    "Red soil": {
        "N": 42.0,    # Deficient in humus and nitrogen
        "P": 68.0,    # Leached phosphorus
        "K": 78.0,    # Rich in potash & iron oxides
        "ph": 5.8,    # Moderately acidic
        "soil_desc": "Porous Red Sandy-Loam (Good drainage, ideal for pulses, groundnuts, millets)"
    }
}

print("Soil Nutrient Benchmark Profiles:")
for soil_type, prof in SOIL_NUTRIENT_MAPPING.items():
    print(f"• {soil_type:15} -> N: {prof['N']}, P: {prof['P']}, K: {prof['K']}, pH: {prof['ph']}")


## 8. Complete End-to-End Pipeline
**Input:** Soil Photograph + Local Climate Telemetry ($Temperature, Humidity, Rainfall$)  
**Stage 1:** CNN predicts **Soil Type** from photo.  
**Stage 2:** Agronomy Bridge infers **$N, P, K, pH$** profile.  
**Stage 3:** Random Forest recommends the **Best Crop to Cultivate**!


In [ ]:
def predict_crop_from_soil_image(image_path, temperature, humidity, rainfall):
    # 1. Load and Preprocess Image
    img = Image.open(image_path).convert('RGB')
    img_resized = img.resize((224, 224))
    img_array = np.array(img_resized) / 255.0
    img_batch = np.expand_dims(img_array, axis=0)

    # 2. Predict Soil Type via CNN
    soil_preds = soil_model.predict(img_batch, verbose=0)[0]
    best_idx = np.argmax(soil_preds)
    predicted_soil = class_names[best_idx]
    soil_confidence = soil_preds[best_idx] * 100

    # 3. Retrieve Inferred N-P-K & pH Profile
    nutrient_profile = SOIL_NUTRIENT_MAPPING.get(predicted_soil, SOIL_NUTRIENT_MAPPING["Alluvial soil"])
    n = nutrient_profile["N"]
    p = nutrient_profile["P"]
    k = nutrient_profile["K"]
    ph = nutrient_profile["ph"]

    # 4. Predict Recommended Crop via Random Forest
    input_vector = np.array([[n, p, k, temperature, humidity, ph, rainfall]])
    crop_probs = rf_crop_model.predict_proba(input_vector)[0]
    crop_classes = rf_crop_model.classes_
    top_crop_idx = np.argmax(crop_probs)
    recommended_crop = crop_classes[top_crop_idx]
    crop_confidence = crop_probs[top_crop_idx] * 100

    # 5. Display Comprehensive Result
    plt.figure(figsize=(6, 4))
    plt.imshow(img)
    plt.title(f"Predicted Soil: {predicted_soil} ({soil_confidence:.1f}% Match)\nRecommended Crop: {recommended_crop.upper()} ({crop_confidence:.1f}% Confidence)", 
              fontsize=11, fontweight='bold', color='#15803d')
    plt.axis('off')
    plt.show()

    print("=" * 60)
    print(f"📷 1. SOIL IMAGE ANALYSIS RESULT")
    print(f"   • Detected Soil Type : {predicted_soil}")
    print(f"   • Vision Confidence  : {soil_confidence:.2f}%")
    print(f"   • Characteristics    : {nutrient_profile['soil_desc']}")
    print("-" * 60)
    print(f"🧪 2. INFERRED SOIL CHEMICAL PROFILE (Auto-Derived)")
    print(f"   • Nitrogen (N)       : {n} kg/ha")
    print(f"   • Phosphorus (P)     : {p} kg/ha")
    print(f"   • Potassium (K)      : {k} kg/ha")
    print(f"   • Soil Reaction (pH) : {ph} ({'Acidic' if ph < 6.5 else 'Neutral' if ph <= 7.5 else 'Alkaline'})")
    print("-" * 60)
    print(f"🌦️ 3. LOCAL METEOROLOGICAL TELEMETRY")
    print(f"   • Temperature        : {temperature} °C")
    print(f"   • Relative Humidity  : {humidity} %")
    print(f"   • Seasonal Rainfall  : {rainfall} mm")
    print("-" * 60)
    print(f"🌾 4. FINAL PRECISION CROP RECOMMENDATION")
    print(f"   • PRIMARY CROP       : {recommended_crop.upper()}")
    print(f"   • Agronomic Match    : {crop_confidence:.2f}%")
    print("=" * 60)

    return {
        "predicted_soil": predicted_soil,
        "soil_confidence": soil_confidence,
        "inferred_nutrients": {"N": n, "P": p, "K": k, "ph": ph},
        "recommended_crop": recommended_crop,
        "crop_confidence": crop_confidence
    }


## 9. Run a Test Inference
Let's test the entire pipeline on an actual test soil image!


In [ ]:
# Test with a sample test image from test directory
test_sample_cls = "Alluvial soil"
test_sample_dir = os.path.join(TEST_DIR, test_sample_cls)
sample_img_name = os.listdir(test_sample_dir)[0]
test_image_path = os.path.join(test_sample_dir, sample_img_name)

# Run full pipeline with sample weather (Indo-Gangetic monsoon: 25°C, 82% humidity, 240mm rain)
result = predict_crop_from_soil_image(
    image_path=test_image_path,
    temperature=25.0,
    humidity=82.0,
    rainfall=240.0
)
